In [ ]:
#Data Preparation
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import skipgrams
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Lambda, Dense
from tensorflow.keras import backend as K

In [ ]:
# Sample text
corpus = ["the quick brown fox jumped over the lazy dog"]

# Tokenize words
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
word2id = tokenizer.word_index
id2word = {v: k for k, v in word2id.items()}

vocab_size = len(word2id) + 1
print("Vocabulary:", word2id)

# Convert text to sequence of word IDs
seq = tokenizer.texts_to_sequences(corpus)[0]
print("\nText to sequence:", seq)

In [ ]:
#Generate Training Data (Context → Target)
window_size = 2
pairs, labels = skipgrams(sequence=seq, vocabulary_size=vocab_size, window_size=window_size)

print("\nSample context-target pairs (first 5):")
for i in range(5):
    print(f"Context: {id2word[pairs[i][0]]}, Target: {id2word[pairs[i][1]]}")

# Convert to NumPy arrays
pairs = np.array(pairs)
labels = np.array(labels)

In [ ]:
#Build and Train the CBOW Model
embed_dim = 8  # Size of embedding vector

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embed_dim, input_length=2))
model.add(Lambda(lambda x: K.mean(x, axis=1)))  # CBOW averages context embeddings
model.add(Dense(vocab_size, activation='softmax'))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
print("\nTraining CBOW model...")
model.fit(pairs, labels, epochs=200, verbose=0)

In [ ]:
#Output: Word Embeddings and Predictions
weights = model.get_layer('embedding').get_weights()[0]
print("\nWord Embedding shape:", weights.shape)

# Display embeddings for each word
for word, idx in word2id.items():
    print(f"{word}: {weights[idx]}")

# Test prediction
test_context = np.array([[word2id['quick'], word2id['fox']]])
pred = model.predict(test_context)
predicted_word = id2word[np.argmax(pred)]
print("\nPredicted word for context ['quick', 'fox'] →", predicted_word)